In [2]:
import csv
import os
import pandas as pd
from cyvcf2 import VCF

In [3]:
def vcf_parsing(file_path: str) -> str:
    '''
    Parses the VCF file and extracts relevant data, then saves the processed data to a TSV file.

    Args:
    file_path (str): The path to the VCF file for parsing.

    Returns:
    str: A message indicating the creation of the TSV file.
    '''

    folder_path = 'processed_data'
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f'Folder "{folder_path}" created.')
    else:
        print(f'Folder "{folder_path}" exists.')

    # Extract the base name of the file (before .vcf or other extensions)
    basename = os.path.basename(file_path)
    name_without_extension = basename.split('.vcf')[0]

    destination_file = f'{folder_path}/{name_without_extension}_parsed.tsv'
    vcf = VCF(file_path)
    print('VCF file loaded.')
    
    # Define the columns to extract
    info_fields_to_extract = ['AC', 'AC_afr', 'AC_amr', 'AC_nfe',
                              'AC_asj', 'AC_sas', 'AC_eas', 'AC_mid', 'AC_fin',
                              'AN', 'AN_afr', 'AN_amr', 'AN_nfe', 'AN_asj',
                              'AN_sas', 'AN_eas', 'AN_mid', 'AN_fin',
                              'AF', 'AF_afr', 'AF_amr', 'AF_nfe', 'AF_asj',
                              'AF_sas', 'AF_eas', 'AF_mid', 'AF_fin', 'vep']

    vep_field_mapping = {
        1: 'Consequence', 2: 'IMPACT', 3: 'SYMBOL', 4: 'Gene',
        6: 'Feature', 7: 'BIOTYPE', 8: 'EXON', 9: 'INTRON', 
        12: 'cDNA_position', 13: 'CDS_position', 14: 'Protein_position', 15: 'Amino_acids', 16: 'Codons',
        17: 'ALLELE_NUM', 19: 'STRAND', 21: 'VARIANT_CLASS',
        24: 'CANONICAL', 44: 'LoF', 45: 'LoF_filter',
        46: 'LoF_flags', 47: 'LoF_info'
    }

    column_names = ['CHROM', 'POS', 'ID', 'REF', 'ALT', 'AC', 'AC_afr',
                    'AC_amr', 'AC_nfe', 'AC_asj', 'AC_sas', 'AC_eas',
                    'AC_mid', 'AC_fin', 'AN', 'AN_afr', 'AN_amr', 'AN_nfe',
                    'AN_asj', 'AN_sas', 'AN_eas', 'AN_mid', 'AN_fin', 'AF',
                    'AF_afr', 'AF_amr', 'AF_nfe', 'AF_asj', 'AF_sas', 'AF_eas',
                    'AF_mid', 'AF_fin', 'Consequence', 'IMPACT', 'SYMBOL', 'Gene',
                    'Feature', 'BIOTYPE', 'EXON', 'INTRON', 'cDNA_position', 
                    'CDS_position', 'Protein_position', 'Amino_acids', 'Codons',
                    'ALLELE_NUM', 'STRAND', 'VARIANT_CLASS', 'CANONICAL', 
                    'LoF', 'LoF_filter', 'LoF_flags', 'LoF_info']

    print('VCF parsing and streaming to file in progress...')

    with open(destination_file, 'w', newline='') as tsvfile:
        writer = csv.writer(tsvfile, delimiter='\t')
        writer.writerow(column_names)  # Write header
        
        # Iterate over each variant in the VCF file
        for variant in vcf:
            if 'PASS' in variant.FILTERS:
                variant_data = [variant.CHROM, variant.POS,
                                variant.ID, variant.REF, variant.ALT[0]]
                info_data = [variant.INFO.get(field, '.') for field in info_fields_to_extract]
                vep_annotation = variant.INFO.get('vep')

                # Handle multiple transcripts in vep if present
                if vep_annotation:
                    for transcript in vep_annotation.split(','):
                        split_transcript = transcript.split('|')
                        vep_fields = []
                        for key in vep_field_mapping.keys():
                            try:
                                vep_fields.append(split_transcript[key])
                            except Exception:
                                vep_fields.append('.')

                        # Conditions for row selection
                        vep_dict = dict(zip(vep_field_mapping.values(), vep_fields))

                        if (vep_dict.get('VARIANT_CLASS', '.') == 'SNV' and
                            vep_dict.get('Feature', '.').startswith('ENST')):

                            writer.writerow(variant_data + info_data[:-1] + vep_fields)

    print('File writing complete.')
    return f'{destination_file} file created'


In [4]:
vcf_parsing("../gnomad.exomes.v4.1.sites.chr21.vcf.bgz")

Folder "processed_data" exists.
VCF file loaded.
VCF parsing and streaming to file in progress...
File writing complete.


'processed_data/gnomad.exomes.v4.1.sites.chr21_parsed.tsv file created'